# Operando Pipeline Runner

Edit `operando_config.ini`, then run all cells.

This notebook runs the operando correction, background subtraction, event selection, plotting, and video-export workflow using `operando_data.py` and `operando_plots.py`.

Only the experiment neutron-image folder is required. Reference images, electrochemistry, beam history, and background correction are optional.

Both legacy per-run folders and the version-2 layout (`tpx3Files/<experiment>/<run>`, `final/<experiment>/<photon-range>`, and `metadata/<experiment>/<run>`) are supported under either `data/experiment` or `data/experiments`. Timing is read from each run's detailed exposure manifest when present, then from `acquisition.json`/`experiment.json`; raw TPX3 modification times and `tpx_time` are only a legacy fallback. If beam correction is needed, use the GUI timing-range step to confirm that beam history covers the acquisition windows.


## Environment Setup
Load the Python modules used by the workflow and expose the toolbox functions that handle correction, background processing, plotting, and video export.


In [ ]:
# Imports
import os
import sys
import ast
import configparser
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from pathlib import Path
import numpy as np

# Resolve the development modules without importing production operando.
# OPERANDO_DEV_DIR can override discovery when the notebook is copied elsewhere.
notebook_dir = os.path.abspath(os.getcwd())
sys.dont_write_bytecode = True
module_candidates = []
if os.environ.get("OPERANDO_DEV_DIR"):
    module_candidates.append(Path(os.environ["OPERANDO_DEV_DIR"]).expanduser())
module_candidates.extend([
    Path("/home/localadmin/Programs/toolbox/operando_dev"),
    Path("/data01/software/toolbox/operando_dev"),
    Path(r"C:\\Software\\toolbox\\operando_dev"),
    Path.cwd(),
    Path.cwd() / "operando_dev",
    Path.cwd() / "toolbox" / "operando_dev",
])
operando_module_dir = next(
    (path for path in module_candidates if (path / "operando_data.py").is_file()),
    None,
)
if operando_module_dir is None:
    raise FileNotFoundError(
        "Could not find operando_dev. Set OPERANDO_DEV_DIR to its folder."
    )
pipeline_dir = str(operando_module_dir)
data_module_path = str(operando_module_dir / "operando_data.py")
plots_module_path = str(operando_module_dir / "operando_plots.py")
interactive_module_path = str(operando_module_dir / "operando_interactive.py")

# Load operando data module
data_spec = importlib.util.spec_from_file_location("operando_data", data_module_path)
operando_data = importlib.util.module_from_spec(data_spec)
data_spec.loader.exec_module(operando_data)

# Load operando plots module
plots_spec = importlib.util.spec_from_file_location("operando_plots", plots_module_path)
operando_plots = importlib.util.module_from_spec(plots_spec)
plots_spec.loader.exec_module(operando_plots)

run_operando_pipeline = operando_data.run_operando_pipeline
compute_background_image = operando_data.compute_background_image
background_subtract_directory = operando_data.background_subtract_directory


def print_notebook_box(title, lines, marker="="):
    if isinstance(lines, str):
        lines = [lines]
    message_lines = [str(title), *[str(line) for line in lines]]
    width = max([len(line) for line in message_lines] + [72])
    border = str(marker)[:1] * width
    print("")
    print(border)
    for line in message_lines:
        print(line)
    print(border)
    print("")


def print_notebook_warning_box(title, lines):
    print_notebook_box(title, lines, marker="!")


def print_notebook_info_box(title, lines):
    print_notebook_box(title, lines, marker="=")


def print_metadata_match_summary(stage_label, expected_metadata):
    print_notebook_info_box(
        f"{stage_label.upper()} METADATA CHECK PASSED",
        [
            "Metadata match: yes. The existing files were created with the current notebook parameters.",
            "Checked parameters:",
            *[f"  - {key}: {value}" for key, value in expected_metadata.items()],
        ],
    )


## Interactive GUI Controller
Run the next cell to open a step-by-step notebook GUI. Each step shows the same kind of outputs as the linear notebook cells, while keeping path history and run settings.


In [ ]:
# Optional GUI front end for this workflow.
gui_module_path = str(operando_module_dir / "operando_gui.py")
gui_spec = importlib.util.spec_from_file_location("operando_gui", gui_module_path)
operando_gui = importlib.util.module_from_spec(gui_spec)
gui_spec.loader.exec_module(operando_gui)

operando_app = operando_gui.launch_operando_gui(campaign_dir=os.getcwd())
